In [3]:
#install llama-cpp-python on colab w gpu support, optional for gguf models
# !CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python
!pip install evaluate
!pip install rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=84539b6b945828791d6970b74f72542a8bc4fbe612222a89e3d9a0ec24165e18
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [4]:
import sys
sys.set_int_max_str_digits(10000) #for openrouter eval

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
import time
import pandas as pd
import evaluate
import requests
import json

# Google AI Studio imports
try:
    from google import genai
    from google.colab import userdata
    print("Google AI Studio libraries imported.")
except ImportError:
    genai = None
    print("Google AI Studio libraries not found. Run `pip install -U google-generativeai` if needed.")

# google drive import
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ReadmeGenerator

# Detect device
device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"

print(f"Using device: {device}")

# Try to import llama_cpp for GGUF support
try:
    from llama_cpp import Llama
    print("llama-cpp-python is available.")
except ImportError:
    Llama = None
    print("llama-cpp-python is NOT installed. GGUF models will not be supported.")

Google AI Studio libraries imported.
Mounted at /content/drive
[Errno 2] No such file or directory: '/content/drive/MyDrive/ReadmeGenerator'
/content
Using device: cpu
llama-cpp-python is NOT installed. GGUF models will not be supported.


Declaring the Quen3-Coder-Next Model and Loading the Tokenizer

In [ ]:
model_name = "unsloth/Qwen3-4B-Instruct-2507-GGUF"
# Example GGUF usage: model_name = "TheBloke/Llama-2-7B-Chat-GGUF"
# Example Gemini usage: model_name = "aistudio-gemini-1.5-flash"

context_window_len = 90000
max_output_tokens = 5000
tokenizer = None
model = None

if "gguf" in model_name.lower():
    if Llama is None:
        raise ImportError("You requested a GGUF model but llama-cpp-python is not installed. Please install it with `pip install llama-cpp-python`.")

    print(f"Loading GGUF model: {model_name} using llama-cpp-python...")
    # Attempt to load local path or download from Hub
    if os.path.exists(model_name):
        model = Llama(model_path=model_name, n_gpu_layers=-1, verbose=False, n_ctx=context_window_len) # Added n_ctx
    else:
        # Downloads a generic quantized version if not specified
        model = Llama.from_pretrained(
            repo_id=model_name,
            filename="*Q4_K_M.gguf",
            verbose=False,
            n_gpu_layers=-1, # Offload to GPU if available
            n_ctx=context_window_len # Added n_ctx to utilize full context window
        )
    print("GGUF Model loaded successfully!")

elif model_name.startswith("aistudio-"):
    if genai is None:
         raise ImportError("Google Generative AI SDK not installed. Please install it.")

    real_model_name = model_name.replace("aistudio-", "")
    print(f"Configuring Google AI Studio model: {real_model_name}...")

    try:
        GOOGLE_API_KEY = userdata.get('google_aistudio_key')
        client = genai.Client(api_key=GOOGLE_API_KEY)
        print("Google AI Studio Model configured successfully!")
        try:
            for m in client.models.list():
                print(m.name)
        except Exception as e:
            print(f"Error listing models: {e}")
    except Exception as e:
        print(f"Error configuring Google AI Studio: {e}. Make sure 'google_aistudio_key' is set in Colab secrets.")
        model = None

else:
    # Standard Transformers loading
    print(f"Loading Transformers model: {model_name} on {device}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    print("Transformers Model loaded successfully!")

Loading GGUF model: unsloth/Qwen3-4B-Instruct-2507-GGUF using llama-cpp-python...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


./Qwen3-4B-Instruct-2507-Q4_K_M.gguf:   0%|          | 0.00/2.50G [00:00<?, ?B/s]

llama_context: n_ctx_per_seq (90000) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


GGUF Model loaded successfully!


Turncate text utility for preventing errors related to context window overflow

In [5]:
def truncate_text(text, token_limit, encoding=None, default_encoding_name="cl100k_base"):
    """
    Truncate text to a token limit using a universal tiktoken encoding by default.

    Args:
        text (str): Text to truncate.
        token_limit (int): Max tokens allowed.
        encoding: Optional pre-loaded tiktoken Encoding object.
        default_encoding_name (str): Encoding to use if encoding is None.

    Returns:
        str: Truncated text.
    """
    if not text or token_limit <= 0:
        return ""

    if encoding is None:
        try:
            import tiktoken
            encoding = tiktoken.get_encoding(default_encoding_name)
        except Exception as e:
            print(f"Warning: Could not load tiktoken ({e}). Falling back to character-based approximation.")
            chars_per_token = 4
            return text[: token_limit * chars_per_token]

    try:
        tokens = encoding.encode(text)
        if len(tokens) <= token_limit:
            return text
        return encoding.decode(tokens[:token_limit])
    except Exception as e:
        print(f"Error during tokenization/truncation: {e}")
        chars_per_token = 4
        return text[: token_limit * chars_per_token]

README Generation

In [6]:
def generate_readme(parsed_file, model, tokenizer, token_in_limit=16384, token_out_limit=5000):
    # Check if the parsed file exists
    if not os.path.exists(parsed_file):
        print(f"File {parsed_file} does not exist.")
        return None

    with open(parsed_file, 'r', encoding='utf-8', errors='ignore') as f:
        first_line = f.readline()
        code_content = f.read()

    #parse number after 'Tokens'
    file_token_count = int(first_line.split('Tokens:')[1].strip())

    if file_token_count > token_in_limit:
      # Truncate content to token limit
      code_content = truncate_text(code_content, int(token_in_limit*0.7), encoding=tokenizer) #*0.8 for saftey margin since tokenization between models may be different
      print(f"current file token count {file_token_count} exceed limit, truncated")

    # Create a prompt
    prompt = f"Based on the following repository code structure and contents, generate a comprehensive README.md file. Include sections for: Description, Features, Installation, Usage, and any other relevant information.\n\n{code_content}\n\n"

    readme_content = ""

    # --- Inference Logic ---
    # 1. OpenRouter (Triggered by 'or-' prefix in model name)
    if isinstance(model, str) and model.startswith("or-"):
        actual_model_name = model.replace("or-", "")

        print(f"Generating content via OpenRouter model: {actual_model_name} w token out limit: {token_out_limit}")
        try:
            import requests
            response = requests.post(
                url="https://openrouter.ai/api/v1/chat/completions",
                headers={"Authorization": f"Bearer {userdata.get('openrouter_key')}", "Content-Type": "application/json"},
                data=json.dumps({
                    "model": actual_model_name,
                    "messages": [{"role": "user", "content": prompt}],
                    "max_tokens": token_out_limit
                })
            )
            res_json = response.json()
            readme_content = res_json['choices'][0]['message']['content']
        except Exception as e:
            print(f"Error generating with OpenRouter: {e}")
            return ""

    # 1. Google AI Studio (Gemini)
    # We check if 'client' is available in globals, which indicates the aistudio setup was run.
    elif 'client' in globals() and client is not None and (model is None or hasattr(model, 'generate_content')):
        try:
            print(f"Generating content with Google AI Studio model: {real_model_name}...")
            # Using the updated client.models.generate_content syntax
            response = client.models.generate_content(model=f"models/{real_model_name}", contents=prompt)
            readme_content = response.text
        except Exception as e:
            print(f"Error generating content with Gemini: {e}")
            return ""

    # 2. llama-cpp-python
    elif model is not None and hasattr(model, 'create_completion'):
        print("Generating content with llama-cpp...")
        output = model.create_completion(
            prompt,
            max_tokens=token_out_limit,
            temperature=0.7,
            top_p=0.9,
            echo=False
        )
        readme_content = output['choices'][0]['text']

    # 3. Hugging Face Transformers
    elif model is not None and tokenizer is not None:
        print("Generating content with Transformers...")
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=token_in_limit  # Adjusted for potentially large context
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=token_out_limit,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=True,
                temperature=0.7,
                top_p=0.9
            )

        readme_content = tokenizer.decode(outputs[0], skip_special_tokens=True)

    else:
        print("Error: No valid model configuration or client detected.")
        return ""

    # Post-processing
    if "README.md" in readme_content:
        # Attempt to split to get content after the header if the model echoes it
        parts = readme_content.split("README.md", 1)
        if len(parts) > 1:
            readme_content = parts[1].strip()

    # Clean up code blocks if the model wrapped the output
    if readme_content.startswith("```markdown"):
        readme_content = readme_content.replace("```markdown", "", 1)
    elif readme_content.startswith("```"):
        readme_content = readme_content.replace("```", "", 1)

    if readme_content.endswith("```"):
        readme_content = readme_content[:-3]

    return readme_content.strip()

In [ ]:
def save_readme(output_path, readme_content, prefix=""):
    # Saves the generated README.md content to a file in the specified output path
    filename = f"{prefix}generated_README.md" if prefix else "generated_README.md"
    output_path = os.path.join(output_path, filename)

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(readme_content)

    print(f"{prefix}README.md has been saved to {output_path}")
    return output_path

In [ ]:
import os
import time # Ensure time is imported, though it was in a previous cell

def process_all_parsed_files(model, tokenizer, parsed_files_dir, output_path, token_in_limit, token_out_limit):
    print("Starting README generation for all parsed files...")

    # Ensure output directory exists
    if not os.path.exists(output_path):
        os.makedirs(output_path)

    # Iterate through all files in the parsed_files_dir
    for filename in os.listdir(parsed_files_dir):
        if filename.endswith("_parsed_code.txt"): # Assuming parsed files have this extension
            parsed_file_path = os.path.join(parsed_files_dir, filename)
            print(f"Processing: {parsed_file_path}")

            try:
                # Record start time
                time_start = time.time()

                # Generate the README
                readme_content = generate_readme(parsed_file_path, model, tokenizer, token_in_limit, token_out_limit)

                # Record end time
                time_end = time.time()
                time_elapsed = time_end - time_start

                if readme_content:
                    # Extract a prefix for the output filename (e.g., from 'apache_cordova-plugin-splashscreen_parsed_code.txt' to 'apache_cordova-plugin-splashscreen_')
                    prefix = filename.replace('_parsed_code.txt', '_')

                    # Print the generated README content and time taken
                    print("")
                    print(f"Generated README for {filename}:")
                    print(readme_content)
                    print(f"Time taken for {filename}: {time_elapsed:.2f} seconds")
                    print("")

                    # Save the generated README
                    save_readme(output_path, readme_content, prefix=prefix)
                else:
                    print(f"Skipping README generation for {filename} due to empty content.")
            except Exception as e:
                print(f"Error processing file {filename}: {e}")
    print("Finished README generation for all parsed files.")




Generate readmes first

In [ ]:
parsed_file_dir = "/content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator/out"
output_path_baseline = "/content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator/llm_output_baseline"
output_path_finetuned = "/content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator/llm_output_finetuned"
output_path_gemma = "/content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator/gemma_output_baseline"
api_model_baseline = "or-google/gemma-3-27b-it"

if not os.path.exists(output_path_baseline):
    os.makedirs(output_path_baseline)

if not os.path.exists(output_path_finetuned):
    os.makedirs(output_path_finetuned)

if not os.path.exists(output_path_gemma):
    os.makedirs(output_path_gemma)

generate Readme BASELINE GEMMA

In [ ]:
#openrouter doesn't have the same init process as others so hard coding the params
process_all_parsed_files(api_model_baseline, None, parsed_file_dir, output_path_gemma, 90000, 5000)

Starting README generation for all parsed files...
Processing: /content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator/out/pallets_click_parsed_code.txt
current file token count 266763 exceed limit, truncated
Generating content via OpenRouter model: google/gemma-3-27b-it w token out limit: 5000

Generated README for pallets_click_parsed_code.txt:
# Click

[![PyPI version](https://badge.fury.io/py/click.svg)](https://badge.fury.io/py/click)
[![Build Status](https://github.com/pallets/click/workflows/test/badge.svg)](https://github.com/pallets/click/actions)
[![Coverage Status](https://coveralls.io/repos/github/pallets/click/badge.svg?branch=master)](https://coveralls.io/repos/github/pallets/click)
[![License](https://img.shields.io/badge/License-BSD--3--Clause-blue.svg)](https://opensource.org/licenses/BSD-3-Clause)

Click is a Python package for creating beautiful command line interfaces in a composable way with as little code as necessary.

## Description

Click aims to make writin

Generate Readme BASELINE

In [ ]:
process_all_parsed_files(model, tokenizer, parsed_file_dir, output_path_baseline, context_window_len, max_output_tokens)

Streaming output truncated to the last 5000 lines.
## Version

0.12.1

---

## Generated by

This file was generated by a tool. Do not edit it directly.

---

## End of File

This file was generated by a tool. Do not edit it directly.

---

## Version

0.12.1

---

## Generated by

This file was generated by a tool. Do not edit it directly.

---

## End of File

This file was generated by a tool. Do not edit it directly.

---

## Version

0.12.1

---

## Generated by

This file was generated by a tool. Do not edit it directly.

---

## End of File

This file was generated by a tool. Do not edit it directly.

---

## Version

0.12.1

---

## Generated by

This file was generated by a tool. Do not edit it directly.

---

## End of File

This file was generated by a tool. Do not edit it directly.

---

## Version

0.12.1

---

## Generated by

This file was generated by a tool. Do not edit it directly.

---

## End of File

This file was generated by a tool. Do not edit it directly.

---


Generate readmes w FINETUNED model

In [ ]:
model_name = "/content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator/qwen3-4b-instruct-merged-f16.gguf"
# Example GGUF usage: model_name = "TheBloke/Llama-2-7B-Chat-GGUF"
# Example Gemini usage: model_name = "aistudio-gemini-1.5-flash"

context_window_len = 90000
max_output_tokens = 5000
tokenizer = None
model = None

if "gguf" in model_name.lower():
    if Llama is None:
        raise ImportError("You requested a GGUF model but llama-cpp-python is not installed. Please install it with `pip install llama-cpp-python`.")

    print(f"Loading GGUF model: {model_name} using llama-cpp-python...")
    # Attempt to load local path or download from Hub
    if os.path.exists(model_name):
        model = Llama(model_path=model_name, n_gpu_layers=-1, verbose=False, n_ctx=context_window_len) # Added n_ctx
    else:
        # Downloads a generic quantized version if not specified
        model = Llama.from_pretrained(
            repo_id=model_name,
            filename="*Q4_0.gguf",
            verbose=False,
            n_gpu_layers=-1, # Offload to GPU if available
            n_ctx=context_window_len # Added n_ctx to utilize full context window
        )
    print("GGUF Model loaded successfully!")

elif model_name.startswith("aistudio-"):
    if genai is None:
         raise ImportError("Google Generative AI SDK not installed. Please install it.")

    real_model_name = model_name.replace("aistudio-", "")
    print(f"Configuring Google AI Studio model: {real_model_name}...")

    try:
        GOOGLE_API_KEY = userdata.get('google_aistudio_key')
        client = genai.Client(api_key=GOOGLE_API_KEY)
        print("Google AI Studio Model configured successfully!")
        try:
            for m in client.models.list():
                print(m.name)
        except Exception as e:
            print(f"Error listing models: {e}")
    except Exception as e:
        print(f"Error configuring Google AI Studio: {e}. Make sure 'google_aistudio_key' is set in Colab secrets.")
        model = None

else:
    # Standard Transformers loading
    print(f"Loading Transformers model: {model_name} on {device}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    print("Transformers Model loaded successfully!")

Loading GGUF model: /content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator/qwen3-4b-instruct-merged-f16.gguf using llama-cpp-python...


llama_context: n_ctx_per_seq (90000) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


GGUF Model loaded successfully!


In [ ]:
process_all_parsed_files(model, tokenizer, parsed_file_dir, output_path_finetuned, context_window_len, max_output_tokens)

Streaming output truncated to the last 5000 lines.
--- 

> 🚀 **Build powerful, user-friendly tools with Click today.**

--- 

> ✅ **Click is actively maintained and supports Python 3.10+**.

--- 

> 📚 **Documentation**: https://click.palletsprojects.com

--- 

> 🚀 **Start building your CLI today with Click!**

--- 

> ✅ **Click is the go-to library for building robust, user-friendly command-line interfaces in Python.**

--- 

> 🚀 **Build powerful, user-friendly tools with Click today.**

--- 

> ✅ **Click is actively maintained and supports Python 3.10+**.

--- 

> 📚 **Documentation**: https://click.palletsprojects.com

--- 

> 🚀 **Start building your CLI today with Click!**

--- 

> ✅ **Click is the go-to library for building robust, user-friendly command-line interfaces in Python.**

--- 

> 🚀 **Build powerful, user-friendly tools with Click today.**

--- 

> ✅ **Click is actively maintained and supports Python 3.10+**.

--- 

> 📚 **Documentation**: https://click.palletsprojects.com


Updated ReadMe Evals

In [7]:
def calculate_perplexity(text, model_id='distilgpt2'):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
    max_length = model.config.n_positions

    encodings = tokenizer(text, return_tensors='pt', model_max_length=1024, truncation=True) #Note: truncating the readme inputs to 1024, which is not ideal
    stride = 512
    seq_len = encodings.input_ids.size(1)

    nlls = []
    prev_end_loc = 0
    for begin_loc in range(0, seq_len, stride):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc
        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            outputs = model(input_ids, labels=target_ids)
            neg_log_likelihood = outputs.loss

        nlls.append(neg_log_likelihood * trg_len)

        prev_end_loc = end_loc
        if end_loc == seq_len:
            break

    ppl = torch.exp(torch.stack(nlls).sum() / end_loc)
    return ppl.item()

# perplexity_score = calculate_perplexity(generated_readme_content)
# print(f"Perplexity Score: {perplexity_score}")

In [8]:
def get_comprehensive_metrics(gen_text, ref_text, judge_llm_model_name, openrouter_key):
    # 1. Local Metric: ROUGE-L
    rouge = evaluate.load('rouge')
    rouge_results = rouge.compute(predictions=[gen_text], references=[ref_text])
    rouge_l = rouge_results['rougeL']

    # 2. Local Metric: Perplexity
    ppl = calculate_perplexity(gen_text)

    # 3. Consolidated LLM Judge Call (OpenRouter API)
    prompt = f"""You are a technical documentation expert. Compare the GENERATED README against the GROUND TRUTH.

    ### GROUND TRUTH:
    {ref_text}

    ### GENERATED README:
    {gen_text}

    ### TASK:
    Evaluate based on:
    1. Content Accuracy (1-10)
    2. Formatting Quality (1-10)
    3. Technical Completeness (1-10)
    4. Overall Rubric Score (1-5)
    5. Winner: Version A (Generated) or Version B (Ground Truth)

    ### OUTPUT FORMAT (CRITICAL):
    Return ONLY a JSON object with these keys:
    \"accuracy\", \"formatting\", \"completeness\", \"rubric\", \"winner_version\", \"justification\"
    """

    try:
        # OpenRouter API Request
        response = requests.post(
            url="https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {openrouter_key}",
                "Content-Type": "application/json",
            },
            data=json.dumps({
                "model": judge_llm_model_name,
                "messages": [{"role": "user", "content": prompt}],
                "response_format": { "type": "json_object" }
            })
        )

        result = response.json()
        try:
            content = result['choices'][0]['message']['content']
            if not content:
                raise ValueError("Empty content received from OpenRouter")

            clean_json = content.replace('```json', '').replace('```', '').strip()
            llm_data = json.loads(clean_json)
        except Exception as e:
            print(f"Extraction Error: {e}")
            llm_data = {"accuracy": 0, "formatting": 0, "completeness": 0, "rubric": 0, "winner_version": "N/A", "justification": "Error"}

    except Exception as e:
        print(f"OpenRouter Eval Error: {e}")
        llm_data = {"accuracy": 0, "formatting": 0, "completeness": 0, "rubric": 0, "winner_version": "N/A", "justification": "Error"}

    metrics_summary = {
        "Metric": ["ROUGE-L", "Perplexity", "Accuracy (1-10)", "Formatting (1-10)", "Completeness (1-10)", "Rubric (1-5)", "Winner"],
        "Value": [
            round(rouge_l, 4),
            round(ppl, 2),
            llm_data.get('accuracy', 0),
            llm_data.get('formatting', 0),
            llm_data.get('completeness', 0),
            llm_data.get('rubric', 0),
            llm_data.get('winner_version', "N/A")
        ]
    }

    df = pd.DataFrame(metrics_summary)
    return df, llm_data.get('justification', "No justification provided.")


def evaluate_folder(generated_dir, ground_truth_dir, judge_llm_model_name, openrouter_key):
    all_results = []
    gen_files = [f for f in os.listdir(generated_dir) if f.endswith('.md')]

    print(f"Starting evaluation on {len(gen_files)} files using OpenRouter...")

    for filename in gen_files:
        gen_path = os.path.join(generated_dir, filename)
        ref_path = os.path.join(ground_truth_dir, filename.replace('_generated', ''))

        if not os.path.exists(ref_path):
            continue

        with open(gen_path, 'r', encoding='utf-8') as f:
            gen_content = f.read()
        with open(ref_path, 'r', encoding='utf-8') as f:
            ref_content = f.read()

        df_metrics, justification = get_comprehensive_metrics(
            gen_content, ref_content, judge_llm_model_name, openrouter_key
        )

        row_data = dict(zip(df_metrics['Metric'], df_metrics['Value']))

        # Classifies anything other than the generated readme as B
        winner_output = str(row_data['Winner']).strip().lower()
        if 'ground' in winner_output or 'truth' in winner_output or 'ground truth' == winner_output:
            row_data['Winner'] = "B"

        row_data['Filename'] = filename
        row_data['Justification'] = justification
        all_results.append(row_data)

    master_df = pd.DataFrame(all_results)

    numeric_cols = ["ROUGE-L", "Perplexity", "Accuracy (1-10)", "Formatting (1-10)", "Completeness (1-10)", "Rubric (1-5)"]
    summary_stats = master_df[numeric_cols].mean().to_frame().reset_index()
    summary_stats.columns = ['Metric', 'Mean Score']

    # Updated Win Rate Logic to handle 'A' or 'Version A'
    valid_wins = master_df[master_df['Winner'] != "N/A"]
    win_count = valid_wins['Winner'].apply(lambda x: str(x).strip() in ["A", "Version A", "Version A (Generated)"]).sum()
    win_rate = (win_count / len(valid_wins)) * 100 if len(valid_wins) > 0 else 0

    return master_df, summary_stats, win_rate

def save_eval_results_to_txt(summary_stats, master_df, win_rate, filepath="eval_report.txt"):
    """
    Writes evaluation results to a formatted .txt file.
    """
    with open(filepath, 'w', encoding='utf-8') as f:
        # Header for Aggregate Summary
        f.write('='*100 + '\n')
        f.write("Finetuned local model aggregate summary\n")
        f.write('-'*100 + '\n')
        # to_string() preserves the table alignment in text files
        f.write(summary_stats.to_string(index=False) + '\n\n')
        f.write(f"win rate against ground truth: {win_rate:.2f}%\n")

        # Header for Individual Results
        f.write('='*100 + '\n')
        f.write("Finetuned local model metrics per readme\n")
        f.write('-'*100 + '\n')
        # We print the master_df, including the filename and scores
        f.write(master_df.to_string(index=False) + '\n')
        f.write('='*100 + '\n')

    print(f"Results successfully saved to {filepath}")

Compute Eval Metrics for finetuned vs non-finetuned models against baseline

In [9]:
judge_llm_model_name = "qwen/qwen3-32b"
finetuned_generated_readme_dir = "/content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator/llm_output_finetuned"
gt_readme_dir = "/content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator/out"
baseline_generated_readme_dir = "/content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator/llm_output_baseline"
gemma_baseline_generated_readme_dir = "/content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator/gemma_output_baseline"

In [10]:
#finetuned
finetuned_results, finetuned_summary, finetuned_win_rate = evaluate_folder(finetuned_generated_readme_dir, gt_readme_dir, judge_llm_model_name, userdata.get('openrouter_key'))

Starting evaluation on 12 files using OpenRouter...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
print('='*100)
print("Finetuned local model aggregate summary")
print('-'*100)
print(finetuned_summary)
print(f"win rate against ground truth: {finetuned_win_rate}")
print('='*100)
print("Fintuned local model metrics per readme")
print('-'*100)
print(finetuned_results)
ft_eval_path = "/content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator/eval_results/finetuned_eval_report.txt"
save_eval_results_to_txt(finetuned_summary, finetuned_results, finetuned_win_rate, ft_eval_path)

Finetuned local model aggregate summary
----------------------------------------------------------------------------------------------------
                Metric  Mean Score
0              ROUGE-L    0.085367
1           Perplexity   24.133333
2      Accuracy (1-10)    3.166667
3    Formatting (1-10)    2.500000
4  Completeness (1-10)    2.750000
5         Rubric (1-5)    1.666667
win rate against ground truth: 0.0
Fintuned local model metrics per readme
----------------------------------------------------------------------------------------------------
    ROUGE-L  Perplexity  Accuracy (1-10)  Formatting (1-10)  \
0    0.0884       16.75                8                  7   
1    0.1190        3.62                2                  2   
2    0.1209       22.18                7                  8   
3    0.0298        2.33                1                  1   
4    0.0370       13.20                4                  2   
5    0.1181       17.75                3                  1 

In [ ]:
#baseline
baseline_results, baseline_summary, baseline_win_rate = evaluate_folder(baseline_generated_readme_dir, gt_readme_dir, judge_llm_model_name, userdata.get('openrouter_key'))

Starting evaluation on 12 files using OpenRouter...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
print('='*100)
print("Baseline local model aggregate summary")
print('-'*100)
print(baseline_summary)
print(f"win rate against ground truth: {baseline_win_rate}")
print('='*100)
print("Baseline local model metrics per readme")
print('-'*100)
print(baseline_results)
baseline_eval_path = "/content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator/eval_results/baseline_eval_report.txt"
save_eval_results_to_txt(baseline_summary, baseline_results, baseline_win_rate, baseline_eval_path)

Baseline local model aggregate summary
----------------------------------------------------------------------------------------------------
                Metric  Mean Score
0              ROUGE-L    0.100050
1           Perplexity   15.073333
2      Accuracy (1-10)    5.416667
3    Formatting (1-10)    4.666667
4  Completeness (1-10)    3.833333
5         Rubric (1-5)    2.500000
win rate against ground truth: 0.0
Baseline local model metrics per readme
----------------------------------------------------------------------------------------------------
    ROUGE-L  Perplexity  Accuracy (1-10)  Formatting (1-10)  \
0    0.0183       14.30                5                  2   
1    0.1282        4.66                6                  5   
2    0.0600       20.26                6                  3   
3    0.0883       17.20                3                  4   
4    0.0588       17.42                7                  4   
5    0.1270       14.18                8                  9  

In [16]:
#gemma baseline
gemma_results, gemma_summary, gemma_win_rate = evaluate_folder(gemma_baseline_generated_readme_dir, gt_readme_dir, judge_llm_model_name, userdata.get('openrouter_key'))

Starting evaluation on 13 files using OpenRouter...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
print('='*100)
print("Baseline Gemma model aggregate summary")
print('-'*100)
print(gemma_summary)
print(f"win rate against ground truth: {gemma_win_rate}")
print('='*100)
print("Baseline Gemma model metrics per readme")
print('-'*100)
print(gemma_results)
gemma_eval_path = "/content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator/eval_results/gemma_baseline_eval_report.txt"
save_eval_results_to_txt(gemma_summary, gemma_results, gemma_win_rate, gemma_eval_path)

Baseline Gemma model aggregate summary
----------------------------------------------------------------------------------------------------
                Metric  Mean Score
0              ROUGE-L    0.125577
1           Perplexity   16.513077
2      Accuracy (1-10)    6.692308
3    Formatting (1-10)    6.692308
4  Completeness (1-10)    6.307692
5         Rubric (1-5)    3.230769
win rate against ground truth: 0.0
Baseline Gemma model metrics per readme
----------------------------------------------------------------------------------------------------
    ROUGE-L  Perplexity  Accuracy (1-10)  Formatting (1-10)  \
0    0.2108       19.67                7                  7   
1    0.1312       19.67                7                  8   
2    0.1887       17.65                7                  7   
3    0.1471       12.30                6                  7   
4    0.1235       14.59                7                  6   
5    0.1241       12.30                7                  8  

In [ ]:
def evaluate_readmes(reference_content, generated_content, client):
    prompt = f"""Please evaluate and compare the following two versions of a README.md file for the same repository.

    ### Reference README Content:
    {reference_content}

    ### Generated README Content:
    {generated_content}

    ### Evaluation Criteria:
    1. **Content Accuracy**: How well does the generated version match the technical facts and source details in the reference?
    2. **Formatting Quality**: Evaluate the markdown structure, header hierarchy, and overall readability.
    3. **Technical Completeness**: Check the coverage of essential sections like Description, Features, Installation, and Usage.

    ### Instructions:
    - Provide a score (1-10) for each of the three metrics above.
    - Provide a final summary of findings highlighting strengths and weaknesses of the generated version compared to the reference.
    """

    try:
        print(f"Sending evaluation prompt to model: {real_model_name}...")
        response = client.models.generate_content(
            model=f"models/{real_model_name}",
            contents=prompt
        )
        return response.text
    except Exception as e:
        print(f"Error during evaluation: {e}")
        return None

print("Function 'evaluate_readmes' defined successfully.")

Function 'evaluate_readmes' defined successfully.


Evaluating the Two READMEs

Loading the rouge score through comparing generated and actual README

In [ ]:
rouge = evaluate.load('rouge')
results = rouge.compute(predictions=[generated_readme_content], references=[actual_readme_content])
rouge_l_score = results['rougeL']

print(f"ROUGE-L Score: {rouge_l_score}")

ROUGE-L Score: 0.5278875713658322


Calculating Perplexity Score

Grading the Two READMEs with a Rubric

In [ ]:
def grade_with_rubric(actual_readme, generated_readme, client):
    prompt = f"""You are a technical documentation expert. Please grade the following generated README based on its alignment with the provided source code structure.
    \n\n### Rubric (Scale 1-5):\n- 1 (Poor): Inaccurate, missing major technical details, or hallucinates features not in the source.
    \n- 2 (Fair): Significant omissions or several technical inaccuracies.
    \n- 3 (Good): Generally accurate and covers most major features, but lacks depth or has minor errors.
    \n- 4 (Very Good): Accurate, comprehensive, and well-aligned with the source code.
    \n- 5 (Excellent): Perfectly aligned with source code, highly comprehensive, and technically precise.\n\n###
    Source Code Context:\n{actual_readme}\n\n### Generated README Content:\n{generated_readme}\n\n###
    Instructions:\nProvide a numeric score (1-5) and a detailed justification for the grade."""

    try:
        print(f"Evaluating README with rubric using model: {real_model_name}...")
        response = client.models.generate_content(model=f"models/{real_model_name}", contents=prompt)
        return response.text
    except Exception as e:
        print(f"Error during rubric grading: {e}")
        return None

# Runs Rubric Evaluation
# rubric_evaluation = grade_with_rubric(actual_readme_content, generated_readme_content, client)

# if rubric_evaluation:
#     print("\n--- Rubric Evaluation Results ---\n")
#     print(rubric_evaluation)

#     score_search = re.search(r'Grade:\s*(\d)', rubric_evaluation)
#     if score_search:
#         rubric_score = int(score_search.group(1))
#     else:
#         rubric_score = "N/A"
#     print(f"\nExtracted Rubric Score: {rubric_score}")


Judging if Generated README is Superior than Actual README

In [ ]:
def run_win_rate_analysis(generated, reference, client):
    prompt = f"""You are a judge comparing two versions of a README.md file for the same software project.
    \n\n### Version A (Generated):\n{generated}\n\n### Version B (Ground Truth/Reference):\n{reference}\n\n### Task:\n
    Which version is better for a developer using this library? Consider clarity, completeness, and modern documentation standards.\n\n###
    Instructions:\n1. Provide a brief comparison.\n2. State clearly which one wins: 'Winner: Version A' or 'Winner: Version B'."""

    try:
        print(f"Performing Win Rate analysis using model: {real_model_name}...")
        response = client.models.generate_content(model=f"models/{real_model_name}", contents=prompt)
        return response.text
    except Exception as e:
        print(f"Error during win rate analysis: {e}")
        return None

# Runs Win Rate Analysis
win_rate_result = run_win_rate_analysis(generated_readme_content, actual_readme_content, client)

if win_rate_result:
    print("\n--- Win Rate Analysis Result ---\n")
    print(win_rate_result)
    win_status = "Win" if "Winner: Version A" in win_rate_result else "Loss"
    print(f"\nOutcome for Generated README: {win_status}")
else:
    win_status = "N/A"

Performing Win Rate analysis using model: gemini-2.5-flash...

--- Win Rate Analysis Result ---

### Comparison

**Version A (Generated)** is a comprehensive and well-structured README that excels in clarity, completeness, and adherence to modern documentation standards. It starts with a clear description, features list, and an explicit statement about supported platforms, including crucial historical context for platform support. The usage and configuration sections are meticulously detailed, providing clear explanations for each preference, including nuanced notes about `SplashScreenDelay` and `FadeSplashScreenDuration`. A standout feature is the dedicated "Splash Screen Image Configuration" section, which addresses a common pain point with clear examples and directory structure. It also includes important "Contributing" and "License" sections.

**Version B (Ground Truth/Reference)** is functional but less complete and less organized than Version A. Its initial description is brief, 

Overall Score Breakdown

In [ ]:
report_data = {
    "Metric": ["ROUGE-L Score", "Perplexity", "Rubric Score (1-5)", "Win Rate Analysis Outcome"],
    "Value": [
        f"{rouge_l_score:.4f}",
        f"{perplexity_score:.4f}",
        f"{rubric_score}",
        f"{win_status}"
    ],
    "Interpretation": [
        "Measures overlap with ground truth (Higher is better)",
        "Measures linguistic fluency (Lower is better)",
        "Expert judge alignment with source code",
        "Comparison against ground truth baseline"
    ]
}

# Create a DataFrame for better visualization
evaluation_df = pd.DataFrame(report_data)

print("=== Consolidated README Evaluation Report ===\n")
print(evaluation_df.to_string(index=False))

print("\n--- Detailed Summary ---")
print(f"The generated README achieved a ROUGE-L score of {rouge_l_score:.4f} and a Perplexity of {perplexity_score:.4f}.")
print(f"The LLM Judge assigned a Rubric Score of {rubric_score}/5, highlighting high technical accuracy to current code.")
print(f"In the head-to-head battle, the outcome was a '{win_status}' due to specific functional omissions noted by the judge.")

=== Consolidated README Evaluation Report ===

                   Metric  Value                                        Interpretation
            ROUGE-L Score 0.5279 Measures overlap with ground truth (Higher is better)
               Perplexity 8.7669         Measures linguistic fluency (Lower is better)
       Rubric Score (1-5)      3               Expert judge alignment with source code
Win Rate Analysis Outcome    Win              Comparison against ground truth baseline

--- Detailed Summary ---
The generated README achieved a ROUGE-L score of 0.5279 and a Perplexity of 8.7669.
The LLM Judge assigned a Rubric Score of 3/5, highlighting high technical accuracy to current code.
In the head-to-head battle, the outcome was a 'Win' due to specific functional omissions noted by the judge.
